# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### VectorStoreManager

In [4]:
import os
import json
import chromadb
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

class QueryResult:
    def __init__(self, documents, distances):
        self.documents = documents
        self.distances = distances


class VectorStoreManager:
    def __init__(
        self,
        collection_name="udaplay",
        db_path="chromadb",
        embedding_model="text-embedding-3-small",
    ):
        self.embedding_model = embedding_model

        self.openai_client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url="https://openai.vocareum.com/v1",
        )

        self.chroma_client = chromadb.PersistentClient(path=db_path)
        self.collection = self.chroma_client.get_or_create_collection(
            name=collection_name
        )

    def build_game_document(self, game):
        return (
            f"[{game['Platform']}] {game['Name']} "
            f"({game['YearOfRelease']}) - {game['Description']}"
        )

    def embed_text(self, text=[]):
        response = self.openai_client.embeddings.create(
            model=self.embedding_model,
            input=[*text]
        )
        return response.data[0].embedding

    def index_game(self, game, doc_id):
        content = self.build_game_document(game)
        embedding = self.embed_text(content)

        self.collection.upsert(
            ids=[doc_id],
            documents=[content],
            metadatas=[game],
            embeddings=[embedding],
        )

    def index_games_from_folder(self, data_dir="games"):
        for file_name in sorted(os.listdir(data_dir)):
            if not file_name.endswith(".json"):
                continue

            file_path = os.path.join(data_dir, file_name)
            with open(file_path, "r", encoding="utf-8") as f:
                game = json.load(f)

            doc_id = os.path.splitext(file_name)[0]
            self.index_game(game, doc_id)
            print(f"Indexed {doc_id}: {game['Name']}")

    def search_collection(self, query = [], n_results=5):
        query_embedding = self.embed_text([*query])
        return self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results
        )

    def search_games(self, query_texts=[], n_results=5):
        results = self.search_collection([*query_texts], n_results=n_results)

        ids = results.get("ids", [[]])[0]
        docs = results.get("documents", [[]])[0]
        metas = results.get("metadatas", [[]])[0]
        distances = results.get("distances", [[]])[0]

        formatted_results = []
        for i, doc_id in enumerate(ids):
            formatted_results.append({
                "id": doc_id,
                "document": docs[i] if i < len(docs) else None,
                "metadata": metas[i] if i < len(metas) else None,
                "distance": distances[i] if i < len(distances) else None,
            })

        return formatted_results

    def search_games_tool(self, query=[], n_results=5):
        results = self.search_games(query, n_results=n_results)

        if not results:
            return "No matching games found."

        lines = []
        for i, result in enumerate(results, 1):
            metadata = result.get("metadata") or {}
            name = metadata.get("Name", "Unknown")
            platform = metadata.get("Platform", "Unknown")
            year = metadata.get("YearOfRelease", "Unknown")
            distance = result.get("distance", "N/A")
            document = result.get("document", "")

            lines.append(
                f"{i}. {name} ({year}) on {platform} | distance={distance}\n"
                f"   {document}"
            )

        return "\n\n".join(lines)

    def query(self, query_texts=[], n_results=5) :
        results = self.search_games(query_texts, n_results=n_results)

        if not results:
            return []

        documents=[]
        distances=[]
        metadatas=[]

        for result in results:
            distance = result.get("distance", "N/A")
            document = result.get("document", "")
            metadata = result.get("metadata", {})

            documents.append(document)
            distances.append(distance)
            metadatas.append(metadata)

        return dict(
            documents=[documents],
            distances=[distances],
            metadata=[metadatas]
        )
    def test() :
        print('OK Tested')



In [5]:
from lib.tools.vector_store_manager import VectorStoreManager

vs = VectorStoreManager()
vs.index_games_from_folder()

Indexed 001: Gran Turismo
Indexed 002: Grand Theft Auto: San Andreas
Indexed 003: Gran Turismo 5
Indexed 004: Marvel's Spider-Man
Indexed 005: Marvel's Spider-Man 2
Indexed 006: Pokémon Gold and Silver
Indexed 007: Pokémon Ruby and Sapphire
Indexed 008: Super Mario World
Indexed 009: Super Mario 64
Indexed 010: Super Smash Bros. Melee
Indexed 011: Wii Sports
Indexed 012: Mario Kart 8 Deluxe
Indexed 013: Kinect Adventures!
Indexed 014: Minecraft
Indexed 015: Halo Infinite


In [6]:
from lib.tools.vector_store_manager import VectorStoreManager

vs = VectorStoreManager()
response = vs.query(
    "A multiplayer racing game for Nintendo platforms",
    n_results=3
)

print(response)

{'documents': [["[PlayStation 4] Marvel's Spider-Man (2018) - An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.", '[Super Nintendo Entertainment System (SNES)] Super Mario World (1990) - A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.', '[Xbox 360] Kinect Adventures! (2010) - A collection of mini-games designed to showcase the capabilities of the Kinect motion sensor.']], 'distances': [[1.2696553468704224, 1.2696553468704224, 1.2696553468704224]], 'metadata': [[{'Name': "Marvel's Spider-Man", 'Genre': 'Action-adventure', 'Publisher': 'Sony Interactive Entertainment', 'Platform': 'PlayStation 4', 'Description': 'An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.', 'YearOfRelease': 2018}, {'Name': 'Super Mario World', 'Description': 'A classic platformer where Mario embarks on a quest to save Princ